In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

import sys
sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import uniprot_mapping

### Abundance

In [ ]:
excel = pd.read_excel("../../data/Abundance_Geiger2012/mcp.M111.014050-3.xlsx", sheet_name='supp Table 3', header=0, skiprows=[0], skipfooter=4)
excel = excel.loc[:, ['Uniprot', 'iBAQ HEK293_1', 'iBAQ HEK293_2', 'iBAQ HEK293_3', 'iBAQ U2OS_1', 'iBAQ U2OS_2', 'iBAQ U2OS_3']]
excel = excel[~excel.iloc[:, 1:].mean(axis=1).isna()].copy()
excel['HEK293_median'] = excel.iloc[:, 1:4].median(axis=1)
excel['U2OS_median'] = excel.iloc[:, 4:7].median(axis=1)
excel = excel.loc[(~excel['HEK293_median'].isnull()) & (~excel['U2OS_median'].isnull()), ['Uniprot', 'HEK293_median', 'U2OS_median']].copy()
excel = excel.dropna()

In [ ]:
id2HEK293 = {}
id2U2OS = {}
for row in excel.values:
    for pro in row[0].split(";"):
        if pro in id2HEK293.keys():
            id2HEK293[pro].append(row[1])
            id2U2OS[pro].append(row[2])
        else:
            id2HEK293[pro] = [row[1]]
            id2U2OS[pro] = [row[2]]
genes = sorted(id2HEK293.keys())

In [ ]:
mapping_dict = uniprot_mapping(
    from_db="UniProtKB_AC-ID", to_db="Gene_Name", ids=genes
)

In [ ]:
HEK293_dict = {}
for k,v in id2HEK293.items():
    if k in mapping_dict.keys():
        new_k = mapping_dict[k]
        if new_k in HEK293_dict.keys():
            HEK293_dict[new_k].extend(v)
        else:
            HEK293_dict[new_k] = v

In [ ]:
U2OS_dict = {}
for k,v in id2U2OS.items():
    if k in mapping_dict.keys():
        new_k = mapping_dict[k]
        if new_k in U2OS_dict.keys():
            U2OS_dict[new_k].extend(v)
        else:
            U2OS_dict[new_k] = v

In [ ]:
HEK293 = []
for k,v in HEK293_dict.items():
    HEK293.append([k, np.median(v)])

In [ ]:
U2OS = []
for k,v in U2OS_dict.items():
    U2OS.append([k, np.median(v)])

In [ ]:
df = pd.merge(pd.DataFrame(HEK293), pd.DataFrame(U2OS), left_on=0, right_on=0)
df['diff'] = (df['1_x'] - df['1_y']).abs()
df = df.sort_values('diff', ascending=False)
df.columns = [0,1,2,3]

In [ ]:
valid_genes = df[(df[2]/df[1]>1.1) | (df[1]/df[2]>1.1)][0]

In [ ]:
with open("../../result/reference/U2OS-HEK293T_abundance.genes", "w") as f:
    f.write("\n".join(valid_genes.tolist()))

In [ ]:
with open("../../result/reference/U2OS-HEK293T_abundance.background", "w") as f:
    f.write("\n".join(sorted(df[0])))